## **제주 관광객 카드 이용 데이터 API 연동 및 데이터 파이프라인 구축**
___
<br>

### **- 프로젝트 개요**

본 프로젝트는 공공데이터 **API를 활용**하여 제주 관광객의 카드 이용 데이터를 **자동으로 수집**하고, <br>
수집된 데이터에 대한 품질 검증 및 정제 과정을 거쳐 분석 가능한 형태로 저장하는
간단한 데이터 파이프라인을 구축한다.

### **- 프로젝트 구성**

- Open API 연동 및 데이터 수집
- API Pagination을 활용한 전체 데이터 자동 수집
- JSON 응답 데이터의 DataFrame 변환
- 데이터 건수 및 품질 검증
- 결측치 및 중복 데이터 확인
- 데이터 타입 정제
- Raw / Final 데이터 분리 저장
- 실행 과정 및 결과 Logging

### **- 사용 데이터**

- **데이터명:** 제주특별자치도_내국인 관광객 지역, 업종, 성별, 연령대별 카드 이용 데이터
- **데이터 제공:** 제주데이터허브 (https://jejudatahub.net/data/view/data/597)
- **수집 방식:** Open API
- **데이터 형식:** JSON
- **주요 분석 기준:** 지역, 업종, 성별, 연령대, 월별
- **주요 지표:** 이용자 수, 이용 건수, 이용 금액

### **- 데이터 파이프라인**

```text
API 요청
   ↓
JSON 데이터 수신
   ↓
Pagination을 통한 전체 데이터 수집
   ↓
DataFrame 변환
   ↓
데이터 품질 검증
   ├── 데이터 건수 검증
   ├── 결측치 검증
   └── 중복 데이터 검증
   ↓
데이터 타입 정제
   ↓
Raw / Final 데이터 저장
   ↓
실행 로그 저장
```

### **- 프로젝트 구조**
```text
project/
│
├── jeju_card_api_data_pipeline.ipynb
│
├── data/
│   ├── raw/
│   └── output/
│
├── logs/
│
└── docs/
    └── api_guide.png
```

### **- API 가이드**

본 프로젝트에서는 제주데이터허브에서 제공하는
「내국인 관광객, 지역, 업종, 성별, 연령대별 카드 이용 데이터 API」를 활용한다.

아래는 해당 API의 요청 URL 및 Request Parameters를 정리한 공식 API 가이드 문서이다.

<img src="./docs/api_guide.png" width="600">

### **1. 라이브러리 Import**

API 호출, 데이터 처리, 파일 관리 및 실행 로그에 필요한 라이브러리를 불러온다.

In [1]:
# ============================================================
# 라이브러리 Import
# ============================================================

import os # 폴더 및 파일 경로 관리
import time # API 호출 간격 조절
import logging # 실행 로그 관리
from datetime import datetime # 수집시간 및 파일명 기록

import pandas as pd # 데이터 처리

import requests # API 호출

### **2. 프로젝트 폴더 설정**

수집 데이터, 최종 데이터, 실행 로그를 각각 별도의 폴더에 저장한다.

In [2]:
# ============================================================
# 프로젝트 폴더 설정
# ============================================================

BASE_DIR = os.getcwd()

# 원본 데이터
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")

# 정제 및 최종 데이터
OUTPUT_DIR = os.path.join(BASE_DIR, "data", "output")

# 실행 로그
LOG_DIR = os.path.join(BASE_DIR, "logs")

# 폴더가 존재하지 않으면 자동 생성
for directory in [RAW_DIR, OUTPUT_DIR, LOG_DIR]:
    os.makedirs(directory, exist_ok=True)

print("프로젝트 폴더 설정 완료")
print(f"BASE_DIR : {BASE_DIR}")
print(f"RAW_DIR  : {RAW_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"LOG_DIR : {LOG_DIR}")

프로젝트 폴더 설정 완료
BASE_DIR : C:\Users\PC\project\jeju_card_api_pipeline
RAW_DIR  : C:\Users\PC\project\jeju_card_api_pipeline\data\raw
OUTPUT_DIR : C:\Users\PC\project\jeju_card_api_pipeline\data\output
LOG_DIR : C:\Users\PC\project\jeju_card_api_pipeline\logs


### **3. 실행 로그 설정**

데이터 수집 및 처리 과정과 실행 결과를 로그 파일에 기록할 수 있도록 설정한다.

In [3]:
# ============================================================
# 실행 로그 설정
# ============================================================

# 실행 시각을 기준으로 로그 파일명 생성, 로그 파일 저장 경로 설정
log_filename = datetime.now().strftime("api_pipeline_%Y%m%d_%H%M%S.log")
log_path = os.path.join(LOG_DIR, log_filename)

# 로그 출력 형식 및 저장 위치 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_path, encoding="utf-8"), # 로그 파일 저장
        logging.StreamHandler() # Jupyter 실행 화면에도 로그 출력
    ]
)
# 현재 프로젝트 전용 logger 생성
logger = logging.getLogger(__name__)

# 파이프라인 시작 로그 기록
logger.info("API 데이터 수집 파이프라인 시작")

2026-09-29 03:21:13,090 | INFO | API 데이터 수집 파이프라인 시작


### **4. API 기본 설정**

제주데이터허브 API 인증정보와 Endpoint 및 데이터 조회 조건을 설정한다.

In [4]:
# ============================================================
# 제주데이터허브 API 설정
# ============================================================

# API 인증키는 코드에 직접 입력하지 않고 환경변수에서 불러온다.
# 인증정보가 코드에 노출되는 것을 방지하고, 환경에 따라 키를 변경할 수 있도록 한다.
APP_KEY = os.getenv("JEJU_DATAHUB_APP_KEY")

if not APP_KEY:
    raise ValueError( "JEJU_DATAHUB_APP_KEY 환경변수가 설정되지 않았습니다.")

# API Endpoint
# 웹 서비스나 API에 접근하기 위한 특정 URL 또는 웹 주소
API_URL = (
    "https://open.jejudatahub.net/"
    "api/proxy/"
    "5D5a577taba7tbb71at1b1bt9tatata9/"
    f"{APP_KEY}"
)

print("API Endpoint 설정 완료")

API Endpoint 설정 완료


In [5]:
# ============================================================
# API 요청 조건
# ============================================================

START_DATE = "201801"
END_DATE = "202012"

# 페이지 번호
PAGE_NUMBER = 1

# 한 번에 가져올 데이터 수. 페이지별 컨텐츠 수
# API 가이드상 최대 100
LIMIT = 100

params = {
    "startDate": START_DATE,
    "endDate": END_DATE,
    "number": PAGE_NUMBER,
    "limit": LIMIT
}

print("API 요청 조건")
print(f"조회기간 : {START_DATE} ~ {END_DATE}")
print(f"페이지   : {PAGE_NUMBER}")
print(f"건수     : {LIMIT}")

API 요청 조건
조회기간 : 201801 ~ 202012
페이지   : 1
건수     : 100


### **5. API 연결 테스트, 응답 확인, JSON 변환, 응답 구조 확인**

API 연결 상태를 확인하고 JSON 응답을 변환하여 API가 제공하는 데이터 구조를 확인한다.

In [6]:
# ============================================================
# API 연결 테스트
# ============================================================

try:
    response = requests.get(
        API_URL,
        params=params,
        timeout=30 # API 서버가 30초 동안 응답하지 않으면 요청을 종료한다.
    )
    # HTTP 4xx 또는 5xx 오류가 발생하면 예외를 발생시켜
    # 오류 응답을 정상 데이터로 처리하지 않도록 한다.
    response.raise_for_status()

    logger.info(f"API 연결 성공 | HTTP Status: {response.status_code}")

except requests.exceptions.RequestException as e:
    logger.error(f"API 연결 실패 | {e}")
    raise

2026-09-29 03:21:18,643 | INFO | API 연결 성공 | HTTP Status: 200


In [7]:
# ============================================================
# API 응답 JSON 변환
# ============================================================

try:
    response_json = response.json()
    logger.info("JSON 응답 변환 성공")

except ValueError as e:
    logger.error(f"JSON 변환 실패 | {e}")
    raise

response_json

2026-09-29 03:21:21,265 | INFO | JSON 응답 변환 성공


{'totCnt': 8019,
 'hasMore': True,
 'data': [{'sido': '제주도',
   'sigungu': '제주시',
   'cityGubun': '읍면',
   'marketType': '유흥',
   'userType': '제주도민',
   'ageGroup': '60대이상',
   'gender': '여',
   'dtYearMonth': '201812',
   'userCount': 23,
   'useCount': 29,
   'useCost': 4240100},
  {'sido': '제주도',
   'sigungu': '제주시',
   'cityGubun': '읍면',
   'marketType': '유흥',
   'userType': '제주도민',
   'ageGroup': '60대이상',
   'gender': '남',
   'dtYearMonth': '201812',
   'userCount': 114,
   'useCount': 190,
   'useCost': 27414000},
  {'sido': '제주도',
   'sigungu': '제주시',
   'cityGubun': '읍면',
   'marketType': '유흥',
   'userType': '제주도민',
   'ageGroup': '50대',
   'gender': '여',
   'dtYearMonth': '201812',
   'userCount': 52,
   'useCount': 71,
   'useCost': 6848000},
  {'sido': '제주도',
   'sigungu': '제주시',
   'cityGubun': '읍면',
   'marketType': '유흥',
   'userType': '제주도민',
   'ageGroup': '50대',
   'gender': '남',
   'dtYearMonth': '201812',
   'userCount': 259,
   'useCount': 458,
   'useCost': 914685

In [8]:
# ============================================================
# API 응답 구조 확인
# ============================================================

print("응답 데이터 타입:")
print(type(response_json))

print("\n응답 Key:")
print(response_json.keys()) # API 응답의 최상위 Key 확인

응답 데이터 타입:
<class 'dict'>

응답 Key:
dict_keys(['totCnt', 'hasMore', 'data'])


### **6. 추출 데이터 정보 확인**

API 응답에서 전체 데이터 건수, 현재 페이지 데이터 및 추가 페이지 존재 여부를 확인한다.

In [9]:
# ============================================================
# API 응답 데이터 추출
# ============================================================

total_count = response_json["totCnt"]
has_more = response_json["hasMore"]
data = response_json["data"]

print(f"전체 데이터 건수 : {total_count:,}")
print(f"현재 페이지 건수 : {len(data):,}")
print(f"추가 페이지 존재 : {has_more}")

전체 데이터 건수 : 8,019
현재 페이지 건수 : 100
추가 페이지 존재 : True


### **7. 전체 페이지 데이터 수집, 변환, 기본 정보 확인**

Pagination을 활용하여 전체 데이터를 수집하고 DataFrame으로 변환한 후 데이터의 기본 구조를 확인한다.

In [10]:
# ============================================================
# 전체 데이터 페이지네이션 수집
# ============================================================

all_data = []
page = 1
limit = 100

# API가 제공하는 전체 데이터 건수 초기화
api_total_count = None

while True:
    params = {
        "startDate": START_DATE,
        "endDate": END_DATE,
        "number": page,
        "limit": limit
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()
    result = response.json()

    # API가 제공하는 전체 데이터 건수를 저장한다.
    # 이후 실제 수집 건수와 비교하여 전체 데이터가 정상적으로 수집되었는지 검증한다.
    if api_total_count is None:
        api_total_count = result["totCnt"]

    # 현재 페이지 데이터 추가
    all_data.extend(result["data"])

    print(
        f"페이지 {page} 수집 완료 | "
        f"누적 {len(all_data):,}건"
    )

    # 다음 페이지가 없으면 종료
    # API가 제공하는 hasMore 값을 기준으로 다음 페이지 존재 여부를 판단한다.
    # 따라서 전체 페이지 수를 코드에 직접 입력할 필요가 없다.
    if not result["hasMore"]:
        break

    page += 1
    time.sleep(0.2) # 연속적인 API 요청을 잠시 쉬어 서버에 과도한 요청이 발생하지 않도록 한다.

print()
print(f"API 전체 데이터 건수 : {api_total_count:,}")
print(f"실제 수집 데이터 건수 : {len(all_data):,}")

페이지 1 수집 완료 | 누적 100건
페이지 2 수집 완료 | 누적 200건
페이지 3 수집 완료 | 누적 300건
페이지 4 수집 완료 | 누적 400건
페이지 5 수집 완료 | 누적 500건
페이지 6 수집 완료 | 누적 600건
페이지 7 수집 완료 | 누적 700건
페이지 8 수집 완료 | 누적 800건
페이지 9 수집 완료 | 누적 900건
페이지 10 수집 완료 | 누적 1,000건
페이지 11 수집 완료 | 누적 1,100건
페이지 12 수집 완료 | 누적 1,200건
페이지 13 수집 완료 | 누적 1,300건
페이지 14 수집 완료 | 누적 1,400건
페이지 15 수집 완료 | 누적 1,500건
페이지 16 수집 완료 | 누적 1,600건
페이지 17 수집 완료 | 누적 1,700건
페이지 18 수집 완료 | 누적 1,800건
페이지 19 수집 완료 | 누적 1,900건
페이지 20 수집 완료 | 누적 2,000건
페이지 21 수집 완료 | 누적 2,100건
페이지 22 수집 완료 | 누적 2,200건
페이지 23 수집 완료 | 누적 2,300건
페이지 24 수집 완료 | 누적 2,400건
페이지 25 수집 완료 | 누적 2,500건
페이지 26 수집 완료 | 누적 2,600건
페이지 27 수집 완료 | 누적 2,700건
페이지 28 수집 완료 | 누적 2,800건
페이지 29 수집 완료 | 누적 2,900건
페이지 30 수집 완료 | 누적 3,000건
페이지 31 수집 완료 | 누적 3,100건
페이지 32 수집 완료 | 누적 3,200건
페이지 33 수집 완료 | 누적 3,300건
페이지 34 수집 완료 | 누적 3,400건
페이지 35 수집 완료 | 누적 3,500건
페이지 36 수집 완료 | 누적 3,600건
페이지 37 수집 완료 | 누적 3,700건
페이지 38 수집 완료 | 누적 3,800건
페이지 39 수집 완료 | 누적 3,900건
페이지 40 수집 완료 | 누적 4,000건
페이지 41 수집 완료 | 누적 4,100건
페이

In [11]:
# ============================================================
# 수집 데이터 → DataFrame 변환
# ============================================================

df = pd.DataFrame(all_data)

# 이후 데이터 정제 과정에서 df가 변경되더라도
# API에서 처음 수집한 원본 상태를 보존할 수 있도록 복사본을 만든다.
raw_df = df.copy()

print(f"최종 수집 건수 : {len(df):,}건")
print(f"컬럼 수 : {len(df.columns)}개")

최종 수집 건수 : 8,019건
컬럼 수 : 11개


In [12]:
# ============================================================
# 기본 데이터 확인
# ============================================================

print("데이터 크기:", df.shape)

print("\n컬럼명:")
print(df.columns.tolist())

print("\n데이터 타입:")
print(df.dtypes)

print("\n샘플 데이터:")
display(df.head())

데이터 크기: (8019, 11)

컬럼명:
['sido', 'sigungu', 'cityGubun', 'marketType', 'userType', 'ageGroup', 'gender', 'dtYearMonth', 'userCount', 'useCount', 'useCost']

데이터 타입:
sido             str
sigungu          str
cityGubun        str
marketType       str
userType         str
ageGroup         str
gender           str
dtYearMonth      str
userCount      int64
useCount       int64
useCost        int64
dtype: object

샘플 데이터:


,sido,sigungu,cityGubun,marketType,userType,ageGroup,gender,dtYearMonth,userCount,useCount,useCost
0,제주도,제주시,읍면,유흥,제주도민,60대이상,여,201812,23,29,4240100
1,제주도,제주시,읍면,유흥,제주도민,60대이상,남,201812,114,190,27414000
2,제주도,제주시,읍면,유흥,제주도민,50대,여,201812,52,71,6848000
3,제주도,제주시,읍면,유흥,제주도민,50대,남,201812,259,458,91468500
4,제주도,제주시,읍면,유흥,제주도민,40대,여,201812,77,99,10021600


### **8. 데이터 품질 검증 및 정제**

결측치와 중복 데이터를 검증하고 데이터 타입을 정리하여 최종 데이터의 품질을 확보한다.

In [13]:
# ============================================================
# 결측치 검사
# ============================================================

missing = df.isna().sum()

missing = missing[missing > 0]

if len(missing) == 0:
    print("✅ 결측치 없음")
else:
    display(missing)

✅ 결측치 없음


In [14]:
# ============================================================
# 중복 데이터 검사
# ============================================================

duplicate_count = df.duplicated().sum()

print(f"중복 데이터 : {duplicate_count:,}건")

if duplicate_count == 0:
    print("✅ 중복 데이터 없음")
else:
    print("⚠️ 중복 데이터 확인 필요")

중복 데이터 : 0건
✅ 중복 데이터 없음


In [15]:
# ============================================================
# 데이터 타입 확인 및 정리
# ============================================================

# API에서 문자열 형태로 전달될 수 있는 날짜 및 수치형 컬럼을
# 분석에 사용할 수 있는 적절한 자료형으로 변환한다.
# 숫자로 변환할 수 없는 값은 NaN으로 처리하여 이후 품질검증에서 확인한다.

# dtYearMonth → 날짜형
df["dtYearMonth"] = pd.to_datetime(
    df["dtYearMonth"],
    format="%Y%m"
)

# userCount → 숫자형
df["userCount"] = pd.to_numeric(
    df["userCount"],
    errors="coerce"  
)

# useCount → 숫자형
df["useCount"] = pd.to_numeric(
    df["useCount"],
    errors="coerce"
)

# useCost → 숫자형
df["useCost"] = pd.to_numeric(
    df["useCost"],
    errors="coerce"
)

print(df.dtypes)

sido                      str
sigungu                   str
cityGubun                 str
marketType                str
userType                  str
ageGroup                  str
gender                    str
dtYearMonth    datetime64[us]
userCount               int64
useCount                int64
useCost                 int64
dtype: object


In [16]:
# ============================================================
# 최종 데이터 품질검증
# ============================================================

print("=" * 50)
print("DATA QUALITY CHECK")
print("=" * 50)

print(f"전체 행 수       : {len(df):,}")
print(f"전체 컬럼 수     : {len(df.columns)}")
print(f"결측치           : {df.isna().sum().sum():,}")
print(f"중복 데이터      : {df.duplicated().sum():,}")

print("=" * 50)

if (
    len(df) == api_total_count
    and df.isna().sum().sum() == 0
    and df.duplicated().sum() == 0
):
    print("✅ DATA QUALITY CHECK : PASS")
else:
    print("⚠️ DATA QUALITY CHECK : 확인 필요")

DATA QUALITY CHECK
전체 행 수       : 8,019
전체 컬럼 수     : 11
결측치           : 0
중복 데이터      : 0
✅ DATA QUALITY CHECK : PASS


### **9. Raw 데이터 저장, 최종 데이터 저장, 실행 로그 저장**

수집한 원본 데이터와 정제된 최종 데이터를 각각 저장하고 전체 실행 결과를 로그로 기록한다.

In [17]:
# ============================================================
# Raw 데이터 저장
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

raw_path = os.path.join(
    RAW_DIR,
    f"jeju_card_api_raw_{timestamp}.csv"
)

raw_df.to_csv(
    raw_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Raw 데이터 저장 완료")
print(raw_path)

Raw 데이터 저장 완료
C:\Users\PC\project\jeju_card_api_pipeline\data\raw\jeju_card_api_raw_20260929_032228.csv


In [18]:
# ============================================================
# 최종 데이터 저장
# ============================================================

final_path = os.path.join(
    OUTPUT_DIR,
    f"jeju_card_api_final_{timestamp}.csv"
)

df.to_csv(
    final_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"최종 데이터 저장 완료")
print(final_path)

최종 데이터 저장 완료
C:\Users\PC\project\jeju_card_api_pipeline\data\output\jeju_card_api_final_20260929_032228.csv


In [19]:
# ============================================================
# 실행 결과 로그
# ============================================================

logger.info(f"API 전체 데이터 : {api_total_count:,}건")
logger.info(f"실제 수집 데이터 : {len(df):,}건")
logger.info(f"결측치 : {df.isna().sum().sum():,}건")
logger.info(f"중복 데이터 : {df.duplicated().sum():,}건")
logger.info(f"Raw 파일 저장 : {raw_path}")
logger.info(f"최종 파일 저장 : {final_path}")

logger.info("API 데이터 수집 및 품질검증 완료")

2026-09-29 03:22:31,869 | INFO | API 전체 데이터 : 8,019건
2026-09-29 03:22:31,870 | INFO | 실제 수집 데이터 : 8,019건
2026-09-29 03:22:31,871 | INFO | 결측치 : 0건
2026-09-29 03:22:31,875 | INFO | 중복 데이터 : 0건
2026-09-29 03:22:31,875 | INFO | Raw 파일 저장 : C:\Users\PC\project\jeju_card_api_pipeline\data\raw\jeju_card_api_raw_20260929_032228.csv
2026-09-29 03:22:31,876 | INFO | 최종 파일 저장 : C:\Users\PC\project\jeju_card_api_pipeline\data\output\jeju_card_api_final_20260929_032228.csv
2026-09-29 03:22:31,877 | INFO | API 데이터 수집 및 품질검증 완료
